# Phase 2 — Preprocessing & Feature Engineering

Inspects the already-processed outputs in `data/processed/`. The pipeline that generated them:
1. Filtered to `for_sale`, dropped rows missing price/size/zip, applied outlier bounds
2. Engineered `price_per_sqft`, `price_per_acre`, `bed_bath_ratio`, `total_rooms`
3. Extracted ZHVI current value and 1yr/5yr growth rates per zip code
4. Merged Realtor listings onto ZHVI features on `zip_code`

In [1]:
import sys
sys.path.insert(0, '..')

import pandas as pd

from src.utils import CLEANED_REALTOR, ZHVI_FEATURES, MERGED_DATA

## 1. Cleaned Realtor Data

In [2]:
df = pd.read_parquet(CLEANED_REALTOR)
print(f'Shape: {df.shape}')
df.head()

Shape: (856386, 16)


,brokered_by,status,price,bed,bath,acre_lot,street,city,state,zip_code,house_size,prev_sold_date,price_per_sqft,price_per_acre,bed_bath_ratio,total_rooms
0,92147.0,for_sale,95000.0,5.0,2.0,0.16,151105.0,Quebradillas,Puerto Rico,95000,1757.0,NaN,54.069435,593750.000000,2.5,7.0
1,10368.0,for_sale,175000.0,3.0,1.0,60.00,553526.0,Berlin,New York,12022,1176.0,NaN,148.809525,2916.666748,3.0,4.0
2,22006.0,for_sale,425000.0,3.0,2.0,2.02,263302.0,Claverack,New York,12521,1600.0,2021-11-24,265.625000,210396.046875,1.5,5.0
3,48310.0,for_sale,225000.0,4.0,2.0,0.24,871278.0,Copake,New York,12521,1239.0,2018-02-01,181.598068,937500.000000,2.0,6.0
4,7797.0,for_sale,419000.0,3.0,3.0,1.90,286373.0,Copake,New York,12516,1800.0,NaN,232.777771,220526.312500,1.0,6.0


## 2. Engineered Features

In [3]:
df[['price', 'house_size', 'price_per_sqft', 'bed_bath_ratio', 'total_rooms']].describe().round(2)

,price,house_size,price_per_sqft,bed_bath_ratio,total_rooms
count,856386.00,856386.00,856386.00,856386.00,856386.00
mean,570648.88,2078.24,272.70,1.43,5.83
std,753679.69,1177.51,280.76,0.58,2.10
min,10500.00,101.00,1.34,0.08,2.00
25%,235000.00,1332.00,141.20,1.00,5.00
50%,379000.00,1803.00,199.11,1.33,6.00
75%,610000.00,2496.00,298.09,1.50,7.00
max,9999999.00,19915.00,21800.00,15.00,30.00


## 3. ZHVI Features

In [4]:
zhvi = pd.read_csv(ZHVI_FEATURES)
print(f'ZHVI zip codes: {len(zhvi):,}')
zhvi.describe().round(4)

ZHVI zip codes: 26,281


,zip_code,zhvi_current,zhvi_1yr_ago,growth_1yr,growth_5yr
count,26281.0000,2.628100e+04,2.628100e+04,26281.0000,25025.0000
mean,48313.9166,3.658014e+05,3.598235e+05,0.0164,0.3012
std,27395.1839,3.268955e+05,3.178474e+05,0.0514,0.1600
min,1001.0000,2.855241e+04,3.237897e+04,-0.4165,-0.5980
25%,25962.0000,1.927728e+05,1.875404e+05,-0.0103,0.2195
50%,47842.0000,2.824334e+05,2.756615e+05,0.0187,0.3204
75%,70377.0000,4.254619e+05,4.210711e+05,0.0460,0.4018
max,99929.0000,8.078219e+06,7.410641e+06,0.4180,1.1201


## 4. Merged Dataset

In [5]:
merged = pd.read_parquet(MERGED_DATA)
matched = merged['zhvi_current'].notna().sum()
print(f'Properties matched to ZHVI: {matched:,} / {len(merged):,} ({matched/len(merged)*100:.1f}%)')
print(f'Merged shape: {merged.shape}')
merged.head()

Properties matched to ZHVI: 850,781 / 856,386 (99.3%)
Merged shape: (856386, 20)


,brokered_by,status,price,bed,bath,acre_lot,street,city,state,zip_code,house_size,prev_sold_date,price_per_sqft,price_per_acre,bed_bath_ratio,total_rooms,zhvi_current,zhvi_1yr_ago,growth_1yr,growth_5yr
0,92147.0,for_sale,95000.0,5.0,2.0,0.16,151105.0,Quebradillas,Puerto Rico,95000,1757.0,NaN,54.069435,593750.000000,2.5,7.0,NaN,NaN,NaN,NaN
1,10368.0,for_sale,175000.0,3.0,1.0,60.00,553526.0,Berlin,New York,12022,1176.0,NaN,148.809525,2916.666748,3.0,4.0,245358.593423,243545.945065,0.007443,0.258139
2,22006.0,for_sale,425000.0,3.0,2.0,2.02,263302.0,Claverack,New York,12521,1600.0,2021-11-24,265.625000,210396.046875,1.5,5.0,477275.645292,446806.854928,0.068192,0.489499
3,48310.0,for_sale,225000.0,4.0,2.0,0.24,871278.0,Copake,New York,12521,1239.0,2018-02-01,181.598068,937500.000000,2.0,6.0,477275.645292,446806.854928,0.068192,0.489499
4,7797.0,for_sale,419000.0,3.0,3.0,1.90,286373.0,Copake,New York,12516,1800.0,NaN,232.777771,220526.312500,1.0,6.0,460065.545843,431260.572796,0.066793,0.490520
